# 02 一日の予定を最適化する（OR-Tools CP-SAT）

テーマパークで「どの施設に、どの順で、いつ並び、ショーのどの回を見るか」を同時に決め、希望度の合計が最も大きい一日の予定を計算します。

| 考えること | このNotebookでの内容 |
| --- | --- |
| 何を入力するか | 施設の位置と徒歩時間、体験時間、希望度、時間帯別の待ち時間、有料の時間短縮の枠と料金、ショーの公演回、食事・休憩、入退園時刻、予算、必須項目 |
| 何を決めるか | 訪問する施設と順序、並び始める時刻、利用方法（通常／有料の時間短縮）、ショーの公演回、食事の時刻 |
| 何を守るか | 入口から出口までの1本の経路、移動時間、予定が重ならない、時間帯・枠・開演の条件、退園時刻、予算、必須項目 |
| 何を最大化するか | 希望度の合計（同点なら追加費用が少ない予定、さらに同点なら移動時間が短い予定） |

**最初に行う操作**: メニューの「ランタイム → すべてのセルを実行」を選ぶか、上から順に `Shift + Enter` で実行します。CPUのランタイムで動きます（GPU不要）。
標準の「すべて実行」では小さなデモだけを計算します（乱数による一括実験は、設定で選んだときだけ実行します）。

**注意**

- パーク・施設・ショー・待ち時間・料金は、すべて**架空の値**です。実在のテーマパークの施設・待ち時間・料金・制度を再現したものではありません。
- スコアは、入力した**希望度の合計**です。実際の来園者の満足度を測った値ではありません。
- 待ち時間は「並び始めた時刻の時間帯の値」を使う決定論的なモデルです。結果は**この1分単位のモデルの厳密解**であり、現実の将来の待ち時間を保証するものではありません。
- 有料の時間短縮（以下「短縮利用」）は、「入力した枠を取得できる、またはすでに取得している」と仮定して計算します。実際の購入・抽選・在庫確認・アプリの操作は行いません。現在の制度を完全に再現したものでもありません（次に購入できる時刻、売り切れ、無料の案内などは初版では扱いません）。

研究上の問い（このNotebookの後半）: **固定時刻の予定（ショー）があると、どの施設の時間短縮に費用を配分するべきかは、どのように変わるか。**

## 0. セットアップ

次のセルは、(1) GitHub からこのリポジトリを HTTPS で取得し、(2) この作品を本体用の依存（OR-Tools など）付きでインストールし、(3) デモのパークのデータを読み込みます。

- `REPO_REF` は取得するコードの版（ブランチ名・タグ名・コミットSHA）です。開発中は作業ブランチを指定しています。main に反映された後は `"main"` に変えます。**Colab で PR の Notebook を開いても、セットアップが自動的に PR のコードを取得するわけではありません**。
- ローカルのチェックアウト（repoルート・作品フォルダ・notebooks フォルダ）から開いた場合は、そのコードを使います。
- `REPO_REF` を変えて再実行し「ランタイムを再起動」と表示されたら、メニューの「ランタイム → セッションを再起動する」を選び、このセルから実行し直します。
- 2回目以降の実行では、前回取得したコミットをそのまま使います（最新にするには `UPDATE_CHECKOUT = True`）。
- 取得とインストールの後は、計算にインターネット接続は使いません。Colab の `/content/` に置いたファイル（計算結果を含む）は、ランタイムが終わると消えます。

01 の Notebook を先に実行しておく必要はありません。

In [ ]:
# @title セットアップ（最初に1回実行します。中身は読まなくても使えます）
# リポジトリの取得 → 作品のインストール → デモ設定の読み込み を行います。
REPO_URL = "https://github.com/shogo-ishikawa/2026_3Q_M4Th4.git"
REPO_REF = "claude/new-session-cc8kr0"  # @param {type:"string"}
CHECKOUT_DIR = "/content/2026_3Q_M4Th4"  # Colab で取得する場所（Colab 以外では ./2026_3Q_M4Th4）
UPDATE_CHECKOUT = False                  # True にすると、同じ REPO_REF でももう一度取得して最新にする
PROJECT_DIR_OVERRIDE = ""                # 配布zipを展開したフォルダを使う場合に指定（例: "/content/v0.1.0"）
PROJECT_SUBDIR = "projects/park-itinerary-lab"
INSTALL_EXTRA = "main"                  # この Notebook に必要な追加の依存（01: intro、02: main）
NOTEBOOK_VERSION = "0.1.0"               # この Notebook が対応する作品の版

import importlib
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
from importlib import metadata
from pathlib import Path

if sys.version_info < (3, 12):
    raise RuntimeError(f"Python 3.12 以上が必要です（PuLP 4 の要件）。今の版: {platform.python_version()}")

GIT_ENV = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # 認証の入力待ちで止まらないようにする
VERIFIED_VERSIONS = {"pulp": "4.2.0", "cbcbox": "2.929", "ortools": "9.15.6755"}
WATCHED_MODULES = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "google.protobuf": "protobuf"}


def run_git(args, cwd):
    """git コマンドを実行する。失敗したら、理由を表示して止める。"""
    done = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, env=GIT_ENV, timeout=300)
    if done.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} が失敗しました:\n{done.stderr.strip()}")
    return done.stdout.strip()


def git_or_none(args, cwd):
    try:
        return run_git(args, cwd)
    except (RuntimeError, OSError, subprocess.TimeoutExpired):
        return None


def is_project_dir(path):
    pyproject = Path(path) / "pyproject.toml"
    return pyproject.is_file() and 'name = "park-itinerary-lab"' in pyproject.read_text(encoding="utf-8")


def find_local_project(start):
    """今いるフォルダとその親から、この作品のフォルダを探す（repoルート・作品フォルダ・notebooks のどこからでも）。"""
    for folder in [start, *start.parents]:
        for candidate in (folder, folder / PROJECT_SUBDIR):
            if is_project_dir(candidate):
                return candidate.resolve()
    return None


def same_url(a, b):
    def clean(url):
        return url.strip().rstrip("/").removesuffix(".git").lower()
    return clean(a) == clean(b)


def fetch_commit(repo_dir, ref):
    """REPO_REF（ブランチ・タグ・コミットSHA・refs/pull/番号/head）を取得し、コミットSHAを返す。"""
    try:
        run_git(["fetch", "--quiet", "origin", ref], repo_dir)
        return run_git(["rev-parse", "FETCH_HEAD^{commit}"], repo_dir)
    except (RuntimeError, subprocess.TimeoutExpired) as error:
        first_error = error
    if re.fullmatch(r"[0-9a-fA-F]{7,40}", ref):  # 短いSHAは、すべてのブランチとタグを取得してから探す
        run_git(["fetch", "--quiet", "--tags", "origin", "+refs/heads/*:refs/remotes/origin/*"], repo_dir)
        found = git_or_none(["rev-parse", "--verify", "--quiet", f"{ref}^{{commit}}"], repo_dir)
        if found:
            return found
    raise RuntimeError(
        f"REPO_REF = {ref!r} を取得できませんでした（main には切り替えません）。"
        f"ブランチ名・タグ名・SHA とネットワークを確認してください。\n{first_error}"
    )


def prepare_clone(checkout_dir):
    """CHECKOUT_DIR に REPO_REF のコードを用意する。別のリポジトリや保存されていない変更は上書きしない。"""
    if not checkout_dir.exists():
        if git_or_none(["rev-parse", "--show-toplevel"], checkout_dir.parent):
            raise RuntimeError(f"{checkout_dir.parent} は別のGitリポジトリの中です。CHECKOUT_DIR を変えてください。")
        checkout_dir.mkdir(parents=True)
        run_git(["init", "--quiet"], checkout_dir)
        run_git(["remote", "add", "origin", REPO_URL], checkout_dir)
        run_git(["config", "--local", "parklab.managed", "true"], checkout_dir)
    if git_or_none(["config", "--local", "--get", "parklab.managed"], checkout_dir) != "true":
        raise RuntimeError(
            f"{checkout_dir} は、このセルが作ったチェックアウトではないため変更しません。"
            "CHECKOUT_DIR を変えるか、そのチェックアウトの中で Notebook を開いてください。"
        )
    origin = run_git(["remote", "get-url", "origin"], checkout_dir)
    if not same_url(origin, REPO_URL):
        raise RuntimeError(f"{checkout_dir} は別のリポジトリ（{origin}）です。CHECKOUT_DIR を変えてください。")
    head = git_or_none(["rev-parse", "--verify", "--quiet", "HEAD"], checkout_dir)
    last_ref = git_or_none(["config", "--local", "--get", "parklab.ref"], checkout_dir)
    last_commit = git_or_none(["config", "--local", "--get", "parklab.commit"], checkout_dir)
    if not UPDATE_CHECKOUT and head and last_ref == REPO_REF and head == last_commit:
        print(f"前回取得したコミット {head} を使います（最新にするには UPDATE_CHECKOUT = True）。")
        return checkout_dir / PROJECT_SUBDIR, head
    commit = fetch_commit(checkout_dir, REPO_REF)
    if git_or_none(["cat-file", "-e", f"{commit}:{PROJECT_SUBDIR}/pyproject.toml"], checkout_dir) is None:
        raise RuntimeError(
            f"REPO_REF = {REPO_REF!r}（コミット {commit}）には {PROJECT_SUBDIR} がありません。"
            "まだその版に反映されていない可能性があります。README で REPO_REF を確認してください。"
        )
    changes = run_git(["status", "--porcelain"], checkout_dir)
    if head != commit:
        if changes:
            raise RuntimeError(f"{checkout_dir} に保存されていない変更があるため、{REPO_REF} へ切り替えません。\n{changes}")
        run_git(["checkout", "--quiet", "--detach", commit], checkout_dir)
        print(f"チェックアウトを {head or '（なし）'} から {commit} に切り替えました。")
    elif changes:
        print("注意: チェックアウトに保存されていない変更があります（そのまま使います）:\n" + changes)
    run_git(["config", "--local", "parklab.ref", REPO_REF], checkout_dir)
    run_git(["config", "--local", "parklab.commit", commit], checkout_dir)
    return checkout_dir / PROJECT_SUBDIR, commit


def same_sources(installed_dir, source_dir):
    """インストールされたパッケージのファイルが、取得したコードと同じかを調べる。"""
    names = sorted(path.name for path in Path(source_dir).glob("*.py"))
    if names != sorted(path.name for path in Path(installed_dir).glob("*.py")):
        return False
    return all((Path(installed_dir) / n).read_bytes() == (Path(source_dir) / n).read_bytes() for n in names)


def installed_matches(source_dir):
    spec = importlib.util.find_spec("park_itinerary_lab")
    if spec is None or not spec.submodule_search_locations:
        return False
    return same_sources(list(spec.submodule_search_locations)[0], source_dir)


def install_project(project_dir):
    """作品を一時フォルダにコピーしてからインストールする（チェックアウトの中に build/ などを作らない）。"""
    with tempfile.TemporaryDirectory() as temp:
        copy = Path(temp) / "park-itinerary-lab"
        copy.mkdir()
        for name in ("pyproject.toml", "README.md"):
            shutil.copy2(project_dir / name, copy / name)
        shutil.copytree(project_dir / "src", copy / "src", ignore=shutil.ignore_patterns("__pycache__", "*.egg-info", "build"))
        done = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", f"{copy}[{INSTALL_EXTRA}]"],
            capture_output=True, text=True,
        )
    if done.returncode != 0:
        raise RuntimeError("pip install に失敗しました:\n" + done.stdout[-2000:] + done.stderr[-3000:])
    importlib.invalidate_caches()


# 1. コードの場所を決める（指定のフォルダ → ローカルのチェックアウト → REPO_REF の取得 の順）
managed_dir = Path(CHECKOUT_DIR) if Path("/content").is_dir() else Path.cwd() / "2026_3Q_M4Th4"
local_project = Path(PROJECT_DIR_OVERRIDE).resolve() if PROJECT_DIR_OVERRIDE else find_local_project(Path.cwd())
if local_project is not None and not local_project.is_relative_to(managed_dir.resolve()):
    if not is_project_dir(local_project):
        raise RuntimeError(f"{local_project} はこの作品のフォルダではありません。")
    PROJECT_DIR = local_project
    COMMIT = git_or_none(["rev-parse", "HEAD"], PROJECT_DIR)
    print("この環境にある作品フォルダを使います（REPO_REF は使いません）。")
    if COMMIT and run_git(["status", "--porcelain", "--", "."], PROJECT_DIR):
        print("注意: 作品フォルダに、コミットしていない変更があります。")
    if not COMMIT and (PROJECT_DIR / "BUILD_INFO.json").is_file():
        print("配布zipの情報:", (PROJECT_DIR / "BUILD_INFO.json").read_text(encoding="utf-8").strip())
else:
    PROJECT_DIR, COMMIT = prepare_clone(managed_dir)
    print(f"{REPO_URL} の {REPO_REF} を {managed_dir} に用意しました。")
SOURCE_DIR = PROJECT_DIR / "src" / "park_itinerary_lab"
OUTPUT_DIR = PROJECT_DIR / "outputs"
SOURCE_INFO = {"repo_url": REPO_URL, "repo_ref": REPO_REF, "commit": COMMIT, "project_dir": str(PROJECT_DIR)}

# 2. インストール（すでに同じコードが入っていて、必要なライブラリもあれば省略）
if "park_itinerary_lab" in sys.modules:
    loaded_dir = Path(sys.modules["park_itinerary_lab"].__file__).parent
    if not same_sources(loaded_dir, SOURCE_DIR):
        raise RuntimeError(
            "前に読み込んだ park_itinerary_lab が、今のコードと違います。"
            "メニューの「ランタイム → セッションを再起動する」の後、このセルからもう一度実行してください。"
        )
NEEDED = {"intro": ["pulp", "cbcbox"], "main": ["ortools", "matplotlib", "matplotlib_fontja"]}[INSTALL_EXTRA]
loaded_before = {module: getattr(sys.modules[module], "__version__", None) for module in WATCHED_MODULES if module in sys.modules}
if not (installed_matches(SOURCE_DIR) and all(importlib.util.find_spec(m) for m in NEEDED)):
    print("作品をインストールしています（1〜2分かかることがあります）…")
    install_project(PROJECT_DIR)
changed = [m for m, version in loaded_before.items() if version != metadata.version(WATCHED_MODULES[m])]
if changed:
    raise RuntimeError(f"読み込み済みの {changed} の版が変わりました。ランタイムを再起動してから、このセルをもう一度実行してください。")

# 3. 読み込んで、取得したコードと同じかを確かめる
import park_itinerary_lab

if not same_sources(Path(park_itinerary_lab.__file__).parent, SOURCE_DIR):
    raise RuntimeError("読み込んだパッケージが取得したコードと一致しません。ランタイムを再起動して、もう一度実行してください。")
if park_itinerary_lab.__version__ != NOTEBOOK_VERSION:
    raise RuntimeError(
        f"この Notebook は版 {NOTEBOOK_VERSION} 用ですが、取得したコードは版 {park_itinerary_lab.__version__} です。"
        "同じ版の Notebook を開くか、REPO_REF を確認してください。"
    )

print("作品フォルダ (PROJECT_DIR):", PROJECT_DIR)
print("コミット:", COMMIT or "Gitのチェックアウトではありません（配布zipなど）")
print("読み込んだモジュール:", park_itinerary_lab.__file__)
print("Python:", platform.python_version(), "／", platform.platform())
for name in ["park-itinerary-lab", "pandas", *{"intro": ["pulp", "cbcbox"], "main": ["ortools", "matplotlib"]}[INSTALL_EXTRA]]:
    version = metadata.version(name)
    note = "" if VERIFIED_VERSIONS.get(name, version) == version else f"（注意: 確認した版は {VERIFIED_VERSIONS[name]}）"
    print(f"{name}: {version}{note}")

# 4. デモ設定（架空のパークのデータ）を読み込む
from park_itinerary_lab.data import load_scenario

scenario = load_scenario(PROJECT_DIR / "configs" / "demo_park.json")
print("デモ設定:", scenario.meta["title"], "／データの種類:", scenario.meta["source_type"])

## 1. 01（入門）との違い

| | 01 入門（PuLP） | 02 本体（CP-SAT） |
| --- | --- | --- |
| 決めること | 選ぶかどうか（0/1）だけ | 選ぶか、順序、時刻、利用方法、公演回 |
| 時間の扱い | 所要時間の合計 ≤ 上限 | 1分単位の時刻。移動・待ち・開演・退園を守る |
| 待ち時間 | 固定 | 並び始める時刻の時間帯で変わる |
| ソルバー | CBC（PuLP 経由） | OR-Tools CP-SAT（直接） |

同じ「入力・決定変数・制約条件・目的関数」の考え方ですが、02 は経路と時刻を含む**別の問題**です。

## 2. 設定セル（学生が変更する値はここにまとめています）

時刻は `"09:30"` のように書きます。計算では 0:00 からの経過分（例: 9:30 → 570）に直して使います。
時間帯 `"10:00-11:00"` は **10:00 を含み 11:00 を含まない**（[開始, 終了)）という意味です。例えば短縮利用の枠 10:00-11:00 なら、10:59 までに並び始める必要があり、11:00 ちょうどは枠の外です。
退園時刻は「この時刻までに出口に着く」期限で、ちょうど着くのは成立とします。

費用は**1人分**（初版の標準）で、料金と予算を同じ基準で入力します。

- **仮定枠（未取得）**: データにある短縮利用の枠。取得できると仮定し、使った場合だけ今回の追加費用を数えます。
- **取得済み枠**: すでに持っている枠（`acquired_shortcuts`）。追加費用は0円で、既払い額は別に表示します。`must_use` で「必ず使う（True）」か「使うかどうかを任せる（False）」かを必ず指定します。

値を変えたら、**このセルから下を順に**実行し直してください（計算のたびにモデルを作り直すので、前の結果は混ざりません）。

In [ ]:
from park_itinerary_lab.experiment import comparison_cases
from park_itinerary_lab.model import SolverSettings

CONDITIONS = {
    "entry_time": "09:00",          # 入園時刻
    "exit_time": "18:00",           # 退園時刻（この時刻までに出口に着く）
    "season": "busy",               # 季節・混雑: "quiet"（平日）/ "normal"（休日）/ "busy"（大型連休）
    "budget_yen": 2000,             # 短縮利用に使える追加費用の上限（1人分, 円）。0 なら使わない
    "cost_basis": "per_person",     # 費用の基準: 1人分（初版の標準）
    "party_size": 1,                # 人数（グループ総額の参考表示だけに使う）
    "allow_new_shortcuts": True,    # 仮定枠（未取得の短縮利用）を検討するか。False なら使わない
    "required_attractions": [],     # 必ず利用する施設のID 例: ["A1"]
    "required_shows": ["S1"],       # 必ず見るショーのID（公演回はモデルが選ぶ）
    "required_meals": ["M1"],       # 必ず確保する食事・休憩のID
    "preferences": {},              # 希望度の上書き 例: {"A3": 8}
    "preference_rank": None,        # 希望順位で指定する場合 例: ["A1", "A4", "A6"]（1位がいちばん高い）
    "acquired_shortcuts": [],       # 取得済み枠（下の例を参照）
    "shortcut_candidates": None,    # None のまま（後半の比較実験の方法Bで使います）
    "purchase_rules": [],           # 追加の購入ルール（初版は未実装なので空のまま）
}
# 取得済み枠の例（使うときは上の [] をこのリストに置き換えます）:
# [{"id": "OWN1", "attraction": "A2", "window": "13:00-14:00",
#   "residual_wait_min": 5, "paid_yen": 1500, "must_use": False}]

SOLVER = SolverSettings(time_limit_s=20, num_workers=8, random_seed=0)  # time_limit_s は各段階の制限時間（秒）
DOWNLOAD = False                               # True にすると、保存した結果を Colab からダウンロードする

# 後半「研究へ向けた比較実験」の設定
EXPERIMENT_BUDGETS = [0, 1000, 2000]          # 比べる予算（円, 1人分）
EXPERIMENT_SHOW_VARIANTS = [                   # 必須ショーの条件
    {"label": "必須ショーなし", "show_id": "S1", "required": False},
    {"label": "元の公演回（11時・15時）", "show_id": "S1"},
    {"label": "13時の回だけ", "show_id": "S1", "starts": ["13:00"]},
    {"label": "パークの奥で開演", "show_id": "S1", "x": -400, "y": 680},
]
REFERENCE_TIME = None                          # 方法Bの基準時刻。None なら入園時刻
RUN_RANDOM_BATCH = False                       # True にすると、乱数で作った複数のパークで一括実験する（時間がかかる）
BATCH_SEEDS = [1, 2, 3]

## 3. 入力データを確かめる

データの出典の種類・確認日・仮定は、データ（`meta`）に記録しています。デモは架空のデータです。

In [ ]:
import pandas as pd

from park_itinerary_lab import report
from park_itinerary_lab.data import prepare_problem

problem = prepare_problem(scenario, CONDITIONS)
print("データ:", scenario.meta["title"])
print("出典の種類:", scenario.meta["source_type"], "／確認日:", scenario.meta["checked_on"])
for line in scenario.meta["assumptions"]:
    print(" -", line)
print("季節・混雑:", problem.wait_table.label)
for warning in problem.warnings:
    print("注意:", warning)
report.attraction_table(problem)

In [ ]:
print("通常利用の待ち時間（分）: 行は施設、列は並び始める時間帯。空欄はその時間帯に通常利用できないこと")
report.wait_table(problem)

In [ ]:
print("有料の時間短縮の枠（枠の時間帯は、並び始める＝入場する時刻に適用）")
report.shortcut_table(problem)

In [ ]:
display(report.show_table(problem))
display(report.meal_table(problem))

## 4. モデル：決定変数・制約条件・目的関数

コードは `src/park_itinerary_lab/model.py` にあり、制約のまとまりごとに短い関数に分けています。

**時刻の区別**（1つの施設について）: 前の場所を出発 → 移動 → **到着** → （必要なら）**意図的な待機** → **並び始め** → **待ち時間** → **体験開始** → **体験終了**。
ショーは「開演の何分前までに到着するか」が決まっていて、その時刻（集合）から開演までを「開演前の待機」とします。

| まとまり | 関数 | 内容 |
| --- | --- | --- |
| 決定変数 | `add_activity_variables` | 行うか $p_a$、利用案 $y_{a,o}$（通常利用の時間帯・短縮の枠・公演回・食事の時間帯）、並び始め $q_a$。$p_a = \sum_o y_{a,o}$、$y_{a,o}=1$ なら $q_a$ はその範囲内、終了 $e_a = q_a + w_o + d_a$ |
| 経路と移動 | `add_route_constraints` | 入口→活動→出口の1本の経路（AddCircuit）。$a$ の次に $b$ なら $q_b \ge e_a + t_{ab}$。最後は $e_a + t_{a,\mathrm{出口}} \le$ 退園時刻 |
| 予算・必須 | `add_budget_and_required` | $\sum c_o y_{a,o} \le$ 予算、必須の活動は $p_a = 1$ |
| 証明を速くする冗長な制約 | `add_redundant_constraints` | 解は変えずに、最適性の証明を速くする |
| 目的 | `objective_expressions` | 希望度合計 $\sum p_a \cdot$希望度、追加費用、移動時間 |

通常利用の待ち時間 $w_o$ は、**並び始める時刻 $q_a$ が入る時間帯の値**です（計算開始時点の待ち時間を一日中使うのではありません）。
到着後すぐに並ぶか、待ち時間が短くなる時間帯や枠の開始まで待ってから並ぶかも、モデルが選びます。

**優先順位つきの最適化**: (1) 希望度合計を最大化 → 最適性を確認できたら、その値を固定して (2) 追加費用を最小化 → 確認できたら固定して (3) 移動時間を最小化。
ソルバーの近似ギャップによる早期終了は使いません（ギャップの許容値0）。結果の状態は次の5つで表示します: 「最適性確認済み」「実行可能な暫定案」「実行不可能と証明」「未判定」「モデルまたは入力のエラー」。
時間切れで「未判定」になった場合、実行不可能とは表示しません。

## 5. 計算する

In [ ]:
from park_itinerary_lab.model import solve_day_plan

result = solve_day_plan(scenario, CONDITIONS, SOLVER)
print(report.status_text(result))
report.stage_table(result)

In [ ]:
assert result.problem is not None and result.problem.conditions == CONDITIONS, "設定を変えたら、計算のセルから実行し直してください"
if result.has_plan:
    display(report.plan_table(result.validation))
    display(report.summary_table(result, result.validation))
else:
    print("予定は返しません。理由:")
    for message in result.messages:
        print(" -", message)

In [ ]:
if result.has_plan:
    figure = report.plot_timeline(result.validation)
else:
    figure = None
    print("予定がないため、図は作りません。")

## 6. 検証：最適化とは別の計算で確かめる

ソルバーが「最適」と答えても、それだけで正しいとはしません。`src/park_itinerary_lab/validate.py` の検証関数は、返された**訪問順・利用方法・並び始めの時刻だけ**から、
移動・到着・待機・待ち時間・開始・終了・費用・必須条件を最初から計算し直します（待ち時間表の引き方も、最適化とは別に書いています）。
`solve_day_plan()` も返す前にこの検証を行い、問題があれば状態を「エラー」にします。ここでは、その結果を表示して確かめます。

In [ ]:
from park_itinerary_lab.validate import compare_with_solver, validate_plan

if result.has_plan:
    check = validate_plan(result.problem, result.steps)
    mismatches = compare_with_solver(result, check)
    print("制約違反:", check.errors or "なし")
    print("ソルバーの値との不一致:", mismatches or "なし")
    print("再計算した値:", {k: check.metrics[k] for k in ("score", "extra_cost_yen", "travel_min", "queue_wait_min", "idle_min")})
    display(report.segment_table(check).head(12))

## 7. 結果を保存する（CSV / JSON / 図）

保存先は作品の `outputs/` の中の、実行ごとの新しいフォルダ（日時_ラベル）です。前回の結果を上書きしません（`outputs/` は Git の管理対象外です）。
`summary.json` には、条件・計算結果・ソフトウェアの版に加えて、使ったコードの版（`REPO_REF` とコミット）も記録します。
Colab では、保存したフォルダを zip にしてダウンロードできます（設定セルの `DOWNLOAD = True`、または左のファイル一覧から右クリック）。

In [ ]:
import shutil

extra = {"conditions": CONDITIONS, "source": SOURCE_INFO}  # 使ったコードの版も一緒に記録する
if result.has_plan:
    run_folder = report.save_run(OUTPUT_DIR, "day_plan", result, result.validation, figure, extra=extra)
else:
    run_folder = report.save_run(OUTPUT_DIR, "day_plan_no_plan", result, extra=extra)
print("保存したフォルダ:", run_folder)
print("ファイル:", sorted(path.name for path in run_folder.iterdir()))
archive = shutil.make_archive(str(run_folder), "zip", root_dir=run_folder)
print("zip:", archive)
if DOWNLOAD:
    try:
        from google.colab import files  # Colab のときだけ使える
        files.download(archive)
    except Exception as error:  # Colab 以外や、ブラウザの設定でダウンロードできない場合
        print("自動ダウンロードはできませんでした（", type(error).__name__, "）。左のファイル一覧から zip を右クリックしてダウンロードしてください。")
else:
    print("ダウンロードするには、設定セルで DOWNLOAD = True にするか、左のファイル一覧から zip を右クリックしてください。")

## 8. 条件を変えて試す

同じ関数を、条件を変えて呼び直します。計算のたびにモデルを作り直すので、前の計算の結果は混ざりません。

In [ ]:
variants = {
    "設定セルの条件": CONDITIONS,
    "予算0円": {**CONDITIONS, "budget_yen": 0},
    "短縮利用を検討しない": {**CONDITIONS, "allow_new_shortcuts": False},
    "スカイコースター（A1）を必須にする": {**CONDITIONS, "required_attractions": ["A1"]},
    "平日（quiet）": {**CONDITIONS, "season": "quiet"},
}
rows = [report.metrics_row(label, solve_day_plan(scenario, conditions, SOLVER)) for label, conditions in variants.items()]
pd.DataFrame(rows)

## 9. 成立しない条件の例

成り立たない条件では、無理に予定を作らず、理由を説明します。必須項目を勝手に外すこともしません。
事前チェックで明らかな衝突（例えば、退園後に始まるショーを必須にした）を見つけた場合は、その内容を表示します。
事前チェックで原因が見つからず、ソルバーが「成立する予定はない」と証明した場合は、原因を断定せずにそう表示します。

In [ ]:
impossible = {**CONDITIONS, "required_shows": ["S1", "S2"]}  # S2（19:30開演）は 18:00 の退園後
impossible_result = solve_day_plan(scenario, impossible, SOLVER)
print(report.status_text(impossible_result))
report.precheck_table(impossible_result)

In [ ]:
typo = {**CONDITIONS, "budjet_yen": 1000}  # 綴りの誤り
print(report.status_text(solve_day_plan(scenario, typo, SOLVER)))

## 10. 独立な正解器との照合（極小問題）

本体のモデルが正しいかを、まったく別の方法で確かめます。`src/park_itinerary_lab/bruteforce.py` は、CP-SAT を使わずに、
**すべての訪問順・利用方法・並び始めの時刻（1分ごと）**を調べる正解器です。待ち時間が時間帯で変わる場合、「到着後すぐに並ぶ」予定だけを調べると、
待ってから並ぶほうが良い予定を見落とすため、時刻もすべて調べます。計算量が多いので、施設3つ・ショー1つ・90分以内の極小問題（`configs/tiny_check_park.json`）で照合します。

In [ ]:
import json

from park_itinerary_lab.bruteforce import solve_by_enumeration
from park_itinerary_lab.data import scenario_from_dict

tiny_raw = json.loads((PROJECT_DIR / "configs" / "tiny_check_park.json").read_text(encoding="utf-8"))
tiny_rows = []
for budget in (0, 500):
    tiny_conditions = {"entry_time": "09:00", "exit_time": "09:50", "season": "tiny",
                       "budget_yen": budget, "allow_new_shortcuts": True}
    model_result = solve_day_plan(scenario_from_dict(tiny_raw), tiny_conditions, SOLVER)
    oracle = solve_by_enumeration(tiny_raw, tiny_conditions)
    model_triple = tuple(model_result.objective_values.get(k) for k in ("score", "extra_cost", "travel"))
    oracle_triple = (oracle["score"], oracle["extra_cost"], oracle["travel"])
    tiny_rows.append({"予算(円)": budget, "本体の状態": model_result.status_label,
                      "本体（希望度, 追加費用, 移動）": model_triple, "正解器（希望度, 追加費用, 移動）": oracle_triple,
                      "一致": model_triple == oracle_triple})
    assert model_triple == oracle_triple, "本体と正解器の結果が一致しません"
display(pd.DataFrame(tiny_rows))
report.plan_table(model_result.validation)

最後の表（予算500円）では、最初の施設に 9:02 に着いた後、待ち時間が40分の時間帯を避けて **9:15 まで待ってから並び**（待ち時間5分）、その後に短縮利用の枠を使っています。
「待ってから並ぶ」予定も、正解器と本体の両方が同じ値を見つけています。

## 11. 自分のデータに差し替える（CSV / JSON）

入力用のテンプレートは `data/templates/` にあります（記入例つき）。
CSV で入力する場合は、`data/templates/csv/` をコピーして値を書き換え、`load_scenario_csv()` で読み込みます。JSON の場合は `configs/demo_park.json` や `data/templates/scenario_template.json` と同じ形式で書き、`load_scenario()` で読み込みます。
実測や公開情報の値を使うときは、`scenario.json` の `meta` に出典の種類（`source_type`）、確認日（`checked_on`）、仮定（`assumptions`）を書いてください。

In [ ]:
from park_itinerary_lab.data import load_scenario_csv

csv_scenario = load_scenario_csv(PROJECT_DIR / "data" / "templates" / "csv")
csv_conditions = {"entry_time": "10:00", "exit_time": "14:00", "season": "holiday", "budget_yen": 1000,
                  "allow_new_shortcuts": True, "required_shows": ["SH1"], "required_meals": ["L1"]}
csv_result = solve_day_plan(csv_scenario, csv_conditions, SOLVER)
print(report.status_text(csv_result))
report.plan_table(csv_result.validation) if csv_result.has_plan else None

---
# 研究へ向けた比較実験

**問い**: 固定時刻の予定（必須ショー）があると、どの施設の時間短縮に費用を配分するべきかは、どのように変わるか。

同じデータ・同じ制約・同じ評価関数・同じ検証関数を使い、次の2つの方法を比べます（新しい最適化を別に書くことはしません）。

- **方法A（同時最適化）**: 訪問先・順序・短縮利用をまとめて決めます（上で使った `solve_day_plan` そのもの）。
- **方法B（候補を先に絞る方法）**: 先に、基準時刻の待ち時間から決めた規則で「短縮利用の候補の施設」を予算内で選び、それ以外の施設の短縮利用を禁止して、同じモデルで解きます。

方法Bの候補集合は「購入済みの集合」ではありません。方法Aと同じく、**実際に使った枠の料金だけ**を追加費用に数えます。
この比較は「候補を先に絞る判断」の影響を調べるためのものです。方法Bは方法Aの選択肢を減らした問題なので、両方の最適性を確認できたケースでは A ≥ B になるのは当然で、それ自体は発見ではありません。
**差が大きい条件・小さい条件と、その理由**を、予定表を見比べて考えることが目的です。

- 差（A − B）は、**両方の希望度合計の最適性を確認できた場合だけ**計算します（片方が暫定案なら、差にソルバーの打ち切りの影響が混ざるため）。比率は使わず、絶対差を表示します。
- タイムアウト・実行不可能・入力エラーのケースも除外せず、状態とともに表に残します。

In [ ]:
from park_itinerary_lab.experiment import CANDIDATE_RULE

print("方法Bの候補の決め方:")
print(CANDIDATE_RULE)
print("基準時刻:", REFERENCE_TIME or f"入園時刻（{CONDITIONS['entry_time']}）")

In [ ]:
from park_itinerary_lab.experiment import representative_cases, run_comparison, save_comparison

cases = comparison_cases(EXPERIMENT_BUDGETS, EXPERIMENT_SHOW_VARIANTS)
comparison, details = run_comparison(scenario, CONDITIONS, cases, SOLVER, REFERENCE_TIME)
comparison_folder = save_comparison(
    comparison, OUTPUT_DIR, "comparison",
    {"base_conditions": CONDITIONS, "budgets": EXPERIMENT_BUDGETS, "show_variants": EXPERIMENT_SHOW_VARIANTS,
     "reference_time": REFERENCE_TIME or CONDITIONS["entry_time"], "solver": SOLVER.__dict__, "source": SOURCE_INFO},
)
print("保存したフォルダ:", comparison_folder)
comparison[["case_id", "method", "status_label", "optimality_scope", "score", "extra_cost_yen", "travel_min",
            "used_shortcut_attractions", "shortcut_candidates", "b_same_options_as_a", "score_diff_A_minus_B", "diff_note"]]

### 代表的なケースを並べて見る

差が最も大きかったケースと、差がなかったケースについて、方法Aと方法Bの予定表と、方法Bの候補の計算過程を並べて表示します。
どの施設の短縮利用を使ったか、ショーの前後で何を回ったか、どこで待ったかを見比べて、差が生まれた（生まれなかった）理由を考えてみましょう。

In [ ]:
for case_id in representative_cases(comparison):
    print("=" * 30, case_id, "=" * 30)
    for method in ("A", "B"):
        entry = details[case_id][method]
        print(f"方法{method}: {entry['result'].status_label}")
        if entry["report"] is not None:
            display(report.plan_table(entry["report"])[["順番", "名前", "利用方法", "入場枠", "到着", "待機(分)", "並び始め・集合", "待ち(分)", "開始", "終了", "追加費用(円)"]])
    print("方法Bの候補の計算過程:")
    display(pd.DataFrame(details[case_id]["B"]["candidates"]))

### 乱数で作った複数のパークでの一括実験（任意）

設定セルの `RUN_RANDOM_BATCH = True` にしたときだけ実行します（標準の「すべて実行」では実行しません）。施設の配置・希望度・待ち時間を乱数の seed で変えたパークを作り、予算を変えて方法Aと方法Bを比べます。
seed・設定・状態・限界値・最適性の確認状況を含む表を `outputs/` に保存します。時間切れのケースも除外しません。

In [ ]:
from park_itinerary_lab.experiment import run_random_batch

if RUN_RANDOM_BATCH:
    batch_conditions = {"entry_time": CONDITIONS["entry_time"], "exit_time": CONDITIONS["exit_time"], "season": "random",
                        "budget_yen": 0, "allow_new_shortcuts": True, "required_shows": ["SH1"], "required_meals": ["MEAL"]}
    batch = run_random_batch(BATCH_SEEDS, EXPERIMENT_BUDGETS, batch_conditions, SOLVER, REFERENCE_TIME)
    batch_folder = save_comparison(batch, OUTPUT_DIR, "random_batch",
                                   {"seeds": BATCH_SEEDS, "budgets": EXPERIMENT_BUDGETS, "conditions": batch_conditions,
                                    "solver": SOLVER.__dict__, "source": SOURCE_INFO})
    print("保存したフォルダ:", batch_folder)
    display(batch[["case_id", "method", "status_label", "score", "extra_cost_yen", "score_diff_A_minus_B", "diff_note"]])
else:
    print("一括実験は実行していません（RUN_RANDOM_BATCH = False）。")

## まとめと次の課題

- 本体モデルは、訪問先・順序・並び始めの時刻・利用方法・公演回を同時に決め、優先順位つきで最適性を確認しました。
- 検証関数と独立な正解器で、返された予定が制約を満たし、極小問題では最適値が一致することを確かめました。
- 比較実験の結果は、上の表の値をそのまま示しています。差が生まれた条件と理由は、予定表を見比べて考察してください。

次の課題の例: 待ち時間の表を自分で調べた値に差し替える（出典と確認日を記録する）、方法Bの候補の決め方を変える、短縮利用の購入の規則（次に購入できる時刻など）をモデルに加える。
初版の仮定と範囲は `docs/model.md`、研究の進め方は `docs/research-plan.md`、確認した内容は `docs/validation.md` にまとめています。